In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 9
fig_height = 6
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvbWl0aHVubWFuaXZhbm5hbi9wcm9qZWN0cy9iZW5jaG1hcmtpbmdfbG9zc19mdW5jdGlvbnNfZWNnX3JlY29uc3RydWN0aW9uL2Jvb2s='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/usr/lib/python3.12/importlib/_bootstrap.py": 1781873160.0, "/usr/lib/python3.12/importlib/_bootstrap_external.py": 1781873160.0, "/usr/lib/python3.12/zipimport.py": 1781873160.0, "/usr/lib/python3.12/codecs.py": 1781873160.0, "/usr/lib/python3.12/encodings/aliases.py": 1781873160.0, "/usr/lib/python3.12/encodings/__init__.py": 1781873160.0, "/usr/lib/python3.12/encodings/utf_8.py": 1781873160.0, "/usr/lib/python3.12/abc.py": 1781873160.0, "/usr/lib/python3.12/io.py": 1781873160.0, "/usr/lib/python3.12/stat.py": 1781873160.0, "/usr/lib/python3.12/_collections_abc.py": 1781873160.0, "/usr/lib/python3.12/genericpath.py": 1781873160.0, "/usr/lib/python3.12/posixpath.py": 1781873160.0, "/usr/lib/python3.12/os.py": 1781873160.0, "/usr/lib/python3.12/_sitebuiltins.py": 1781873160.0, "/usr/lib/python3.12/__future__.py": 1781873160.0, "/usr/lib/python3.12/warnings.py": 1781873160.0, "/usr/lib/python3.12/importlib/__init__.py": 1781873160.0, "/usr/lib/python3.12/importlib/machinery.py": 17818

In [2]:
#| label: engineering-state
import json
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("..")
queue_path = ROOT / "refine-logs/queue/queue_state.json"
audit_path = ROOT / "results/checkpoint_store/compatibility_audit.json"
catalog_path = ROOT / "results/checkpoint_store/catalog.jsonl"

state = json.loads(queue_path.read_text())
audit = json.loads(audit_path.read_text())
queue_counts = Counter(j.get("status", "unknown") for j in state["jobs"])
catalog_rows = [json.loads(line) for line in catalog_path.read_text().splitlines() if line.strip()]
catalog_counts = Counter(r.get("status", "unknown") for r in catalog_rows)

pd.DataFrame({
    "layer": ["queue"] * len(queue_counts) + ["checkpoint catalog"] * len(catalog_counts),
    "state": list(queue_counts) + list(catalog_counts),
    "count": list(queue_counts.values()) + list(catalog_counts.values())
})

,layer,state,count
0,queue,completed,16
1,queue,running,1
2,queue,pending,463
3,checkpoint catalog,remote_verified,15
4,checkpoint catalog,error,130
5,checkpoint catalog,cached,1


In [3]:
#| label: engineering-contract
contract = audit["contract"]
pd.DataFrame({
    "field": [
        "contract id", "source bundle SHA-256", "state schema SHA-256",
        "sample rate", "units", "normalization",
        "train records", "validation records", "test records"
    ],
    "value": [
        contract["contract_id"], contract["approved_source_bundle_sha256"],
        contract["state_schema_sha256"], contract["preprocessing"]["sample_rate_hz"],
        contract["preprocessing"]["units"], contract["preprocessing"]["normalization"],
        contract["split_content_roots"]["train"]["records"],
        contract["split_content_roots"]["val"]["records"],
        contract["split_content_roots"]["test"]["records"]
    ]
})

,field,value
0,contract id,factorial-v4-content-pinned-20260731
1,source bundle SHA-256,6e262086df8e995d90cea19208f189d78658959641069e...
2,state schema SHA-256,6d350ee6785b98efe72e8e058cb717fd5a40201289e156...
3,sample rate,500
4,units,mV
5,normalization,none
6,train records,17418
7,validation records,2183
8,test records,2198


In [4]:
#| label: checkpoint-storage-accounting
#| tbl-cap: Logical checkpoint volume versus bulky bytes currently retained on disk. Historical error rows are quarantined generations, not eligible inference models.
eligible_ids = {
    model["model_id"] for model in audit["models"] if model.get("compatible")
}
eligible_catalog = [row for row in catalog_rows if row["model_id"] in eligible_ids]
assert len(eligible_catalog) == audit["counts"]["compatible"]
assert all(row["status"] in {"remote_verified", "cached"} for row in eligible_catalog)

catalog_logical_bytes = sum(int(row["size_bytes"]) for row in catalog_rows)
catalog_local_bytes = sum(
    int(row["size_bytes"]) for row in catalog_rows if row.get("local_path")
)
eligible_logical_bytes = sum(int(row["size_bytes"]) for row in eligible_catalog)
historical_rows = [row for row in catalog_rows if row["model_id"] not in eligible_ids]
median_eligible_checkpoint_bytes = int(pd.Series(
    [int(row["size_bytes"]) for row in eligible_catalog]
).median())

display(pd.DataFrame([
    ("All catalogued generations", len(catalog_rows), catalog_logical_bytes),
    ("Current inference-eligible generation", len(eligible_catalog), eligible_logical_bytes),
    ("Historical/quarantined generation", len(historical_rows), sum(int(row["size_bytes"]) for row in historical_rows)),
    ("Bulky checkpoint bytes currently local", None, catalog_local_bytes),
    ("Projected 480-model all-local store", 480, median_eligible_checkpoint_bytes * 480),
    ("One-checkpoint streaming-cache bound", 1, median_eligible_checkpoint_bytes),
], columns=["Storage layer", "Model identities", "Bytes"]))

,Storage layer,Model identities,Bytes
0,All catalogued generations,146.0,8476831102
1,Current inference-eligible generation,16.0,656175424
2,Historical/quarantined generation,130.0,7820655678
3,Bulky checkpoint bytes currently local,NaN,0
4,Projected 480-model all-local store,480.0,19685245440
5,One-checkpoint streaming-cache bound,1.0,41010928


In [5]:
#| label: compatible-inference-readiness
#| tbl-cap: Exact archived-model inference readiness on one real PTB-XL test record. Timings are operational observations, not model-quality endpoints.
import hashlib

inference_root = ROOT / "results/factorial_mixed_level/inference_readiness"
inference_summary_path = inference_root / "summary.json"
inference_csv_path = inference_root / "per_model_inference_readiness.csv"
inference_lead_csv_path = inference_root / "per_model_per_lead_case_metrics.csv"
inference_code_path = ROOT / "scripts/benchmark_factorial_inference_readiness.py"
inference_summary = json.loads(inference_summary_path.read_text())
inference_csv_bytes = inference_csv_path.read_bytes()
inference_lead_csv_bytes = inference_lead_csv_path.read_bytes()

assert hashlib.sha256(inference_csv_bytes).hexdigest() == inference_summary["csv_sha256"]
assert hashlib.sha256(inference_lead_csv_bytes).hexdigest() == inference_summary["per_lead_case_metrics_csv_sha256"]
assert hashlib.sha256(audit_path.read_bytes()).hexdigest() == inference_summary["compatibility_audit_sha256"]
assert hashlib.sha256(inference_code_path.read_bytes()).hexdigest() == inference_summary["benchmark_code_sha256"]

inference_rows = pd.read_csv(inference_csv_path)
inference_lead_rows = pd.read_csv(inference_lead_csv_path)
assert len(inference_rows) == audit["counts"]["compatible"]
assert inference_summary["models_completed"] == len(inference_rows)
assert inference_rows.finite.all() and inference_summary["all_finite"]
assert inference_summary["cache_retained_bytes"] == 0
assert len(inference_lead_rows) == len(inference_rows) * 9
assert inference_lead_rows[["mse", "mae", "pearson", "variance_ratio"]].notna().all().all()

display(pd.DataFrame([
    ("Compatible identities expected", inference_summary["models_expected"]),
    ("Models strictly loaded and executed", inference_summary["models_completed"]),
    ("Finite reconstructions", int(inference_rows.finite.sum())),
    ("Input tensor SHA-256", inference_summary["input_sha256"]),
    ("Prepared input shape", str(inference_summary["prepared_input_shape"])),
    ("Reconstruction shape", str(inference_summary["output_shape"])),
    ("Logical checkpoint bytes traversed", inference_summary["checkpoint_logical_bytes"]),
    ("Checkpoint-cache bytes retained", inference_summary["cache_retained_bytes"]),
], columns=["Readiness gate", "Observed value"]))

display(inference_rows[[
    "model_id", "checkpoint_sha256", "load_and_materialize_seconds",
    "forward_median_seconds", "repeats", "output_mean", "output_std", "finite",
]])

,Readiness gate,Observed value
0,Compatible identities expected,16
1,Models strictly loaded and executed,16
2,Finite reconstructions,16
3,Input tensor SHA-256,f10cde0f89950700ad41a3289c2ef607536acdb7c723ae...
4,Prepared input shape,"[1, 3, 5024]"
5,Reconstruction shape,"[1, 12, 5000]"
6,Logical checkpoint bytes traversed,656175424
7,Checkpoint-cache bytes retained,0


,model_id,checkpoint_sha256,load_and_materialize_seconds,forward_median_seconds,repeats,output_mean,output_std,finite
0,f_1000000_s42,ea2df9939d4438de51c702ddd887edf2ccd57b8e2c0947...,0.781873,0.141575,3,0.001137,0.165322,True
1,f_1000001_s42,b5e7d69c9311da7351f6fc22b42294d8869767e34bac02...,0.234772,0.136465,3,0.001997,0.194937,True
2,f_1000002_s42,f7b35963db704af91c2d3145c8b5870369d9aa5fd43ce9...,0.770497,0.135779,3,0.002002,0.180973,True
3,f_1000003_s42,0d6d1620f831eae098313d8be11ac738f37b202f696aa2...,0.676367,0.135522,3,0.002004,0.172353,True
4,f_1000004_s42,0ffac7734981e1a81bc3471386c57c461e0c2dd1913f9b...,0.741335,0.138038,3,0.001512,0.181877,True
5,f_1000010_s42,78b61440b3ea27c80c2ac8b097fcc3a095445e7bd549ca...,0.917773,0.138047,3,-0.002981,0.155180,True
6,f_1000011_s42,f573fb9a73ba494d49b97a5a806f1426d5c43675a4a877...,0.779164,0.137889,3,-0.000284,0.204787,True
7,f_1000012_s42,4d2512473cff1632bfb21cc35d634b706c11694a379546...,0.697849,0.138765,3,0.000845,0.192185,True
8,f_1000013_s42,025ac2aefc16518d0b0cf965f9bd5946c33e1075280c05...,0.701935,0.138645,3,-0.002340,0.171232,True
9,f_1011012_s42,a55a3f42e986fee995bb7d041e8e3e8606c4af63e9c48c...,0.752357,0.137894,3,-0.003926,0.196440,True


In [6]:
#| label: compatible-inference-index-case-lead-summary
#| tbl-cap: Missing-lead reconstruction diagnostics across the current compatible models for PTB-XL test record 100 only.
lead_order = ["III", "aVR", "aVL", "aVF", "V1", "V3", "V4", "V5", "V6"]
index_case_lead_summary = (
    inference_lead_rows.groupby("lead", as_index=False)
    .agg(
        models=("model_id", "nunique"),
        median_mse=("mse", "median"),
        minimum_mse=("mse", "min"),
        maximum_mse=("mse", "max"),
        median_pearson=("pearson", "median"),
        minimum_pearson=("pearson", "min"),
        maximum_pearson=("pearson", "max"),
        median_variance_ratio=("variance_ratio", "median"),
    )
    .set_index("lead")
    .loc[lead_order]
    .reset_index()
)
display(index_case_lead_summary)

,lead,models,median_mse,minimum_mse,maximum_mse,median_pearson,minimum_pearson,maximum_pearson,median_variance_ratio
0,III,16,0.007633,0.003692,0.017923,0.606734,0.328458,0.800903,1.060755
1,aVR,16,0.009372,0.004230,0.014290,0.826392,0.601232,0.924491,1.213572
2,aVL,16,0.014111,0.010720,0.019679,0.538406,-0.278169,0.732835,1.673542
3,aVF,16,0.007184,0.005124,0.022792,0.654487,0.564542,0.755659,1.490661
4,V1,16,0.030264,0.017984,0.115758,0.264799,-0.683032,0.833943,0.804409
5,V3,16,0.039589,0.026823,0.064754,0.869400,0.792287,0.892733,3.075498
6,V4,16,0.018655,0.012146,0.050870,0.893975,0.746001,0.910369,2.496365
7,V5,16,0.010770,0.008674,0.021743,0.831912,0.794302,0.897521,1.370679
8,V6,16,0.011657,0.007800,0.037545,0.740553,0.604340,0.802553,1.152326


In [7]:
#| label: compatible-inference-index-case-heatmap
#| fig-cap: Per-model, per-missing-lead MSE on one indexed PTB-XL test record. This heatmap diagnoses output behavior and must not be read as cohort-level model ranking.
import plotly.express as px

case_mse = (
    inference_lead_rows.pivot(index="model_id", columns="lead", values="mse")
    .reindex(columns=lead_order)
    .sort_index()
)
case_heatmap = px.imshow(
    case_mse,
    aspect="auto",
    color_continuous_scale="Viridis",
    labels={"x": "Reconstructed missing lead", "y": "Exact model identity", "color": "MSE"},
)
case_heatmap.update_layout(height=520)
case_heatmap.show()

In [8]:
#| label: compatible-model-table
model_rows = []
for m in audit["models"]:
    model_rows.append({
        "model_id": m.get("model_id"),
        "compatible": bool(m.get("compatible")),
        "reason": "; ".join(m.get("reasons", [])) or "approved",
        "checkpoint_sha256": str(m.get("checkpoint_sha256", ""))[:16]
    })
pd.DataFrame(model_rows).sort_values(
    ["compatible", "model_id"], ascending=[False, True]
).head(25)

,model_id,compatible,reason,checkpoint_sha256
0,f_1000000_s42,True,approved,ea2df9939d4438de
1,f_1000001_s42,True,approved,b5e7d69c9311da73
2,f_1000002_s42,True,approved,f7b35963db704af9
3,f_1000003_s42,True,approved,0d6d1620f831eae0
4,f_1000004_s42,True,approved,0ffac7734981e1a8
5,f_1000010_s42,True,approved,78b61440b3ea27c8
6,f_1000011_s42,True,approved,f573fb9a73ba494d
7,f_1000012_s42,True,approved,4d2512473cff1632
8,f_1000013_s42,True,approved,025ac2aefc16518d
9,f_1011012_s42,True,approved,a55a3f42e986fee9


In [9]:
#| label: log-coverage
import re

log_dir = ROOT / "refine-logs/queue/logs"
logs = sorted(log_dir.glob("f_*_s*.log"))
rows = []
for path in logs:
    content = path.read_text(errors="replace")
    epochs = [int(x) for x in re.findall(r"Epoch\s+(\d+)", content)]
    rows.append({
        "model_id": path.stem,
        "bytes": path.stat().st_size,
        "last_epoch_seen": max(epochs) if epochs else None,
        "cuda_oom_mentions": content.lower().count("out of memory"),
        "traceback_mentions": content.count("Traceback")
    })
log_df = pd.DataFrame(rows)
pd.DataFrame({
    "quantity": ["log files", "logs with epoch markers", "OOM mentions", "tracebacks"],
    "value": [len(log_df), log_df.last_epoch_seen.notna().sum() if len(log_df) else 0,
              int(log_df.cuda_oom_mentions.sum()) if len(log_df) else 0,
              int(log_df.traceback_mentions.sum()) if len(log_df) else 0]
})

,quantity,value
0,log files,252
1,logs with epoch markers,23
2,OOM mentions,0
3,tracebacks,235


In [10]:
#| label: compatible-training-summary
#| tbl-cap: Optimization diagnostics for current-contract compatible checkpoints.
diagnostic_root = ROOT / "results/factorial_mixed_level/training_diagnostics"
epoch_curves = pd.read_csv(diagnostic_root / "compatible_epoch_curves.csv")
model_diagnostics = pd.read_csv(
    diagnostic_root / "compatible_model_summary.csv"
)
training_diagnostic_status = json.loads(
    (diagnostic_root / "summary.json").read_text()
)
operational_eta = pd.read_csv(
    diagnostic_root / "operational_eta_by_kernel.csv"
)

model_diagnostics.assign(
    duration_minutes=model_diagnostics.duration_seconds / 60,
    val_mse_reduction_percent=-100 * model_diagnostics.val_mse_relative_change,
)[[
    "model_id", "mmd_kernel", "epochs", "duration_minutes",
    "first_val_mse", "last_val_mse", "val_mse_reduction_percent",
    "cuda_oom_mentions", "traceback_mentions", "nonfinite_mentions",
]]

,model_id,mmd_kernel,epochs,duration_minutes,first_val_mse,last_val_mse,val_mse_reduction_percent,cuda_oom_mentions,traceback_mentions,nonfinite_mentions
0,f_1000000_s42,0,10,20.074931,0.0791,0.0305,61.441214,0,0,0
1,f_1000001_s42,1,10,24.079657,0.1077,0.0384,64.345404,0,0,0
2,f_1000002_s42,2,10,23.081824,0.1071,0.0342,68.067227,0,0,0
3,f_1000003_s42,3,10,26.082670,0.0973,0.0310,68.139774,0,0,0
4,f_1000004_s42,4,10,19.076096,0.0977,0.0316,67.656090,0,0,0
5,f_1000010_s42,0,10,19.070904,0.1567,0.0410,73.835354,0,0,0
6,f_1000011_s42,1,10,19.071474,0.1233,0.0427,65.369019,0,0,0
7,f_1000012_s42,2,10,19.071083,0.1425,0.0406,71.508772,0,0,0
8,f_1000013_s42,3,10,24.080596,0.1669,0.0414,75.194727,0,0,0
9,f_1011012_s42,2,10,21.111653,0.1791,0.0504,71.859296,0,0,0


In [11]:
#| label: compatible-operational-eta
#| tbl-cap: Single-GPU operational ETA from current-contract run durations and the live remaining kernel mix.
eta = training_diagnostic_status["operational_eta"]
display(pd.DataFrame([
    ("Compatible duration samples", training_diagnostic_status["compatible_models"]),
    ("Pending or running jobs", eta["remaining_pending_or_running_jobs"]),
    ("Estimated remaining hours", eta["estimated_remaining_hours"]),
    ("Estimated remaining days", eta["estimated_remaining_days"]),
    ("Estimated completion (UTC)", eta["estimated_completion_utc"]),
    ("Observed-min scenario (hours)", eta["observed_min_scenario_hours"]),
    ("Observed-max scenario (hours)", eta["observed_max_scenario_hours"]),
], columns=["ETA field", "Value"]))

display(operational_eta[[
    "mmd_kernel", "compatible_duration_samples", "observed_min_minutes",
    "observed_median_minutes", "observed_max_minutes", "pending_jobs",
    "running_jobs", "running_elapsed_minutes", "estimated_remaining_minutes",
]])

,ETA field,Value
0,Compatible duration samples,16
1,Pending or running jobs,464
2,Estimated remaining hours,160.430027
3,Estimated remaining days,6.684584
4,Estimated completion (UTC),2026-08-08T01:20:02.382754+00:00
5,Observed-min scenario (hours),141.191534
6,Observed-max scenario (hours),184.485394


,mmd_kernel,compatible_duration_samples,observed_min_minutes,observed_median_minutes,observed_max_minutes,pending_jobs,running_jobs,running_elapsed_minutes,estimated_remaining_minutes
0,0,3,17.067856,19.070904,20.074931,93,0,0.000000,1773.594044
1,1,3,18.066144,19.071474,24.079657,93,0,0.000000,1773.647080
2,2,4,19.071083,22.096738,26.093416,92,0,0.000000,2032.899939
3,3,4,18.077916,22.513326,26.082670,92,0,0.000000,2071.225964
4,4,2,19.076096,21.077292,23.078487,93,1,6.830839,1974.434590


In [12]:
#| label: compatible-validation-mse-curves
#| fig-cap: Validation MSE trajectories from current-contract compatible runs. Total composite loss is not plotted because its scale changes with active loss terms.
import plotly.express as px

fig = px.line(
    epoch_curves,
    x="epoch",
    y="val_mse",
    color="model_id",
    markers=True,
    labels={"val_mse": "Validation MSE", "epoch": "Epoch", "model_id": "Model ID"},
)
fig.update_layout(height=480, legend_title_text="Exact model identity")
fig.show()

In [13]:
#| label: temporal-target-detector-eda
#| tbl-cap: Target-side detector availability and event distributions on all 2,198 PTB-XL test records. This is measured before any reconstruction is evaluated.
target_eda_root = ROOT / "results/factorial_v4/temporal_target_detector_eda"
target_eda_summary_path = target_eda_root / "summary.json"
target_eda_csv_path = target_eda_root / "target_detector_feature_eda.csv"
target_subgroup_csv_path = target_eda_root / "target_detector_subgroup_coverage.csv"
target_cache_metadata_path = (
    ROOT / "results/factorial_v4/temporal_mmd_generation_bound"
    / "_target_ptb_xl_features.json"
)
if all(path.is_file() for path in (
    target_eda_summary_path, target_eda_csv_path, target_subgroup_csv_path
)):
    target_eda_summary = json.loads(target_eda_summary_path.read_text())
    target_cache_metadata = json.loads(target_cache_metadata_path.read_text())
    assert target_eda_summary["target_cache_parquet_sha256"] == target_cache_metadata["parquet_sha256"]
    assert hashlib.sha256(target_cache_metadata_path.read_bytes()).hexdigest() == target_eda_summary["target_cache_metadata_sha256"]
    assert hashlib.sha256((ROOT / "scripts/build_temporal_target_detector_eda.py").read_bytes()).hexdigest() == target_eda_summary["builder_code_sha256"]
    assert hashlib.sha256((ROOT / "data/ptb_xl/ptbxl_database.csv").read_bytes()).hexdigest() == target_eda_summary["ptbxl_metadata_sha256"]
    assert hashlib.sha256(target_eda_csv_path.read_bytes()).hexdigest() == target_eda_summary["csv_sha256"]
    assert hashlib.sha256(target_subgroup_csv_path.read_bytes()).hexdigest() == target_eda_summary["subgroup_csv_sha256"]
    target_detector_eda = pd.read_csv(target_eda_csv_path)
    target_detector_subgroups = pd.read_csv(target_subgroup_csv_path)
    assert len(target_detector_eda) == target_eda_summary["feature_rows"]
    assert len(target_detector_subgroups) == target_eda_summary["subgroup_rows"]
    display(target_detector_eda)
else:
    target_detector_eda = pd.DataFrame()
    target_detector_subgroups = pd.DataFrame()
    display(pd.DataFrame({"status": ["Corrected target-detector EDA is rebuilding"]}))

,lead,clinical_feature,records_total,records_detected,record_detection_coverage,events,events_per_detected_record_median,events_per_detected_record_q25,events_per_detected_record_q75,value_median,value_minimum,value_q01,value_q25,value_q75,value_q99,value_maximum,inter_event_interval_ms_median,inter_event_interval_ms_q05,inter_event_interval_ms_q95
0,V3,P_Amp,2198,2178,0.990901,24725,11.0,10.0,13.0,0.000,-2.505,-0.37500,-0.065,0.061,0.62176,3.239000,818.0,534.0,1161.4
1,V3,QT_Interval_ms,2198,2081,0.946770,19936,10.0,8.0,12.0,454.000,72.000,222.00000,416.000,490.000,602.00000,956.000000,838.0,580.0,1502.0
2,V3,Q_Amp,2198,2167,0.985896,23657,11.0,9.0,13.0,-0.085,-4.192,-0.85276,-0.140,-0.035,0.33344,3.740000,808.0,506.0,1238.0
3,V3,R_Amp,2198,2183,0.993176,26129,12.0,10.0,13.0,0.598,-2.513,-0.22772,0.284,0.986,2.48416,8.517000,804.0,490.0,1118.0
4,V3,S_Amp,2198,2182,0.992721,25862,11.0,10.0,13.0,-0.860,-6.283,-3.36800,-1.292,-0.515,0.00500,3.235000,806.0,492.0,1122.0
5,V3,T_Amp,2198,2126,0.967243,23280,11.0,10.0,12.0,0.330,-2.533,-0.20421,0.175,0.511,1.21521,3.564000,826.0,558.0,1166.0
6,V6,P_Amp,2198,2196,0.999090,25332,11.0,10.0,13.0,0.005,-2.576,-0.43000,-0.045,0.055,0.51138,13.049000,814.0,526.0,1152.0
7,V6,QT_Interval_ms,2198,2107,0.958599,19687,10.0,8.0,12.0,402.000,108.000,174.00000,362.000,456.000,596.00000,790.000000,836.0,572.0,1608.0
8,V6,Q_Amp,2198,2167,0.985896,22772,11.0,9.0,12.0,-0.083,-5.760,-0.60500,-0.140,-0.038,0.35500,7.027000,818.0,522.0,1409.6
9,V6,R_Amp,2198,2197,0.999545,26442,12.0,10.0,13.0,0.944,-3.102,-0.07000,0.661,1.258,2.45059,28.238001,802.0,492.0,1114.0


In [14]:
#| label: temporal-target-detector-coverage
#| fig-cap: Target-side record detection coverage by lead and feature. Reconstruction pairing coverage cannot exceed or be interpreted independently of this baseline.
if target_detector_eda.empty:
    display(pd.DataFrame({"status": ["No corrected target-detector EDA artifact yet"]}))
else:
    target_coverage_plot = px.bar(
        target_detector_eda,
        x="clinical_feature",
        y="record_detection_coverage",
        color="lead",
        barmode="group",
        hover_data=[
            "events", "records_detected", "events_per_detected_record_median",
            "value_q01", "value_median", "value_q99",
            "inter_event_interval_ms_median",
        ],
        labels={
            "clinical_feature": "Target feature",
            "record_detection_coverage": "Records with at least one detected event",
        },
    )
    target_coverage_plot.update_yaxes(range=[0, 1])
    target_coverage_plot.update_layout(height=460)
    target_coverage_plot.show()

In [15]:
#| label: temporal-target-detector-subgroups
#| tbl-cap: Target-detector coverage stratified by raw PTB-XL sex code and transparent age bins. Coverage is record-level; this is not a reconstruction-fairness result.
if target_detector_subgroups.empty:
    display(pd.DataFrame({"status": ["No corrected subgroup detector artifact yet"]}))
else:
    display(target_detector_subgroups.sort_values(
        ["subgroup_variable", "record_detection_coverage", "lead", "clinical_feature"]
    ))

,lead,clinical_feature,subgroup_variable,subgroup,records,records_detected,record_detection_coverage
13,V3,QT_Interval_ms,age_group,age_invalid_>100,34,29,0.852941
41,V3,T_Amp,age_group,age_invalid_>100,34,30,0.882353
11,V3,QT_Interval_ms,age_group,age_80_100,306,275,0.898693
53,V6,QT_Interval_ms,age_group,age_80_100,306,278,0.908497
6,V3,P_Amp,age_group,age_invalid_>100,34,31,0.911765
...,...,...,...,...,...,...,...
42,V6,P_Amp,sex_group,sex_code_0,1132,1131,0.999117
63,V6,R_Amp,sex_group,sex_code_0,1132,1131,0.999117
70,V6,S_Amp,sex_group,sex_code_0,1132,1131,0.999117
64,V6,R_Amp,sex_group,sex_code_1,1066,1066,1.000000


In [16]:
#| label: temporal-morphology-acceptance-gate
#| tbl-cap: Live acceptance state for generation-bound morphology artifacts. Partial accepted models are operational diagnostics, not a factorial comparison.
temporal_root = ROOT / "results/factorial_v4/temporal_mmd_generation_bound"
temporal_status = json.loads((temporal_root / "accepted_summary.json").read_text())
temporal_models = pd.read_csv(temporal_root / "accepted_model_artifacts.csv")
temporal_features = pd.read_csv(temporal_root / "accepted_feature_summary.csv")
temporal_exclusions = pd.read_csv(temporal_root / "excluded_artifacts.csv")

pd.DataFrame([
    ("Current compatible checkpoints", temporal_status["eligible_compatible_models"]),
    ("Accepted evaluated checkpoints", temporal_status["accepted_models"]),
    ("Accepted feature rows", temporal_status["accepted_feature_rows"]),
    ("Excluded stale/invalid artifacts", temporal_status["excluded_artifacts"]),
    ("Complete for factorial inference", temporal_status["complete_for_factorial_inference"]),
    ("Evaluator SHA-256", temporal_status["evaluation_code_sha256"]),
    ("Target-cache SHA-256", temporal_status["target_feature_cache_sha256"]),
], columns=["Gate field", "Value"])

,Gate field,Value
0,Current compatible checkpoints,16
1,Accepted evaluated checkpoints,1
2,Accepted feature rows,12
3,Excluded stale/invalid artifacts,3
4,Complete for factorial inference,False
5,Evaluator SHA-256,1948732602e43ada10c75193dc194230e814e8ff68c4d0...
6,Target-cache SHA-256,26ebc1bd52fe409a985361f51d838210703b78a0a4b35a...


In [17]:
#| label: temporal-morphology-exclusions
#| tbl-cap: Artifacts excluded from the current evaluator generation.
if temporal_exclusions.empty:
    display(pd.DataFrame({"status": ["No exclusions"]}))
else:
    display(temporal_exclusions)

,artifact,reason
0,f_1011012_s42.json,evaluator SHA does not match current evaluator...
1,f_1011013_s42.json,evaluator SHA does not match current evaluator...
2,f_1011014_s42.json,evaluator SHA does not match current evaluator...


In [18]:
#| label: temporal-morphology-coverage
#| fig-cap: Per-model detector/pairing coverage for accepted partial artifacts. Coverage is shown to diagnose measurement failure, not to rank loss masks before grid completion.
if temporal_features.empty:
    display(pd.DataFrame({
        "status": ["No artifact yet passes the current evaluator and digest gate"]
    }))
else:
    import plotly.express as px
    coverage_plot = px.bar(
        temporal_features,
        x="clinical_feature",
        y="record_pair_coverage",
        color="model_id",
        facet_col="lead",
        barmode="group",
        hover_data=[
            "n_records_total", "n_records_real_detected",
            "n_records_recon_detected", "n_records_paired", "n_beats",
        ],
        labels={
            "clinical_feature": "Feature",
            "record_pair_coverage": "Records with a finite paired measurement",
        },
    )
    coverage_plot.update_yaxes(range=[0, 1])
    coverage_plot.update_layout(height=500, legend_title_text="Exact model identity")
    coverage_plot.show()

In [19]:
#| label: temporal-morphology-partial-effect-diagnostics
#| fig-cap: Variance ratios for accepted partial artifacts. The horizontal line marks equal reconstructed/target variance; these are feature-detector diagnostics, not completed factorial effects.
if temporal_features.empty:
    display(pd.DataFrame({"status": ["No accepted feature rows"]}))
else:
    variance_plot = px.bar(
        temporal_features,
        x="clinical_feature",
        y="variance_ratio",
        color="model_id",
        facet_col="lead",
        barmode="group",
        hover_data=[
            "mean_real", "mean_recon", "ba_robust_slope", "record_pair_coverage",
        ],
        labels={
            "clinical_feature": "Feature",
            "variance_ratio": "Detected reconstructed / target variance",
        },
    )
    variance_plot.add_hline(y=1.0, line_dash="dash", line_color="black")
    variance_plot.update_layout(height=500, legend_title_text="Exact model identity")
    variance_plot.show()

    distribution_plot = px.bar(
        temporal_features,
        x="clinical_feature",
        y="distribution_rbf_mmd2",
        color="model_id",
        facet_col="lead",
        barmode="group",
        hover_data=[
            "distribution_rbf_bandwidth", "distribution_mmd_real_samples",
            "distribution_mmd_recon_samples", "record_pair_coverage",
        ],
        labels={
            "clinical_feature": "Feature",
            "distribution_rbf_mmd2": "Biased squared RBF MMD",
        },
    )
    distribution_plot.update_layout(
        height=500, legend_title_text="Exact model identity"
    )
    distribution_plot.show()

    partial_diagnostic = pd.DataFrame([
        ("Accepted models", temporal_features.model_id.nunique()),
        ("Lead-feature rows", len(temporal_features)),
        ("Rows with variance ratio < 1", int((temporal_features.variance_ratio < 1).sum())),
        ("Variance-ratio range", f"{temporal_features.variance_ratio.min():.4f}–{temporal_features.variance_ratio.max():.4f}"),
        ("RBF MMD² range", f"{temporal_features.distribution_rbf_mmd2.min():.6f}–{temporal_features.distribution_rbf_mmd2.max():.6f}"),
        ("Pair-coverage range", f"{temporal_features.record_pair_coverage.min():.2%}–{temporal_features.record_pair_coverage.max():.2%}"),
    ], columns=["Partial diagnostic", "Observed value"])
    display(partial_diagnostic)

    coverage_failures = temporal_features.assign(
        records_without_pair=(
            temporal_features.n_records_total - temporal_features.n_records_paired
        )
    )[[
        "model_id", "lead", "clinical_feature", "n_records_total",
        "n_records_real_detected", "n_records_recon_detected",
        "n_records_paired", "records_without_pair", "record_pair_coverage",
    ]].sort_values("record_pair_coverage")
    display(coverage_failures)

    pairing_diagnostics = temporal_features.assign(
        unmatched_event_fraction=(
            temporal_features.n_unmatched_real_events
            + temporal_features.n_unmatched_recon_events
        ) / (
            temporal_features.n_real_beats_detected
            + temporal_features.n_recon_beats_detected
        ).clip(lower=1)
    )[[
        "model_id", "lead", "clinical_feature", "pairing_tolerance_ms",
        "median_abs_pairing_error_ms", "p95_abs_pairing_error_ms",
        "n_unmatched_real_events", "n_unmatched_recon_events",
        "unmatched_event_fraction",
    ]].sort_values("unmatched_event_fraction", ascending=False)
    display(pairing_diagnostics)

,Partial diagnostic,Observed value
0,Accepted models,1
1,Lead-feature rows,12
2,Rows with variance ratio < 1,11
3,Variance-ratio range,0.0678–1.6526
4,RBF MMD² range,0.010241–0.521378
5,Pair-coverage range,91.81%–99.86%


,model_id,lead,clinical_feature,n_records_total,n_records_real_detected,n_records_recon_detected,n_records_paired,records_without_pair,record_pair_coverage
11,f_1000000_s42,V6,QT_Interval_ms,2198,2107,2145,2018,180,0.918107
5,f_1000000_s42,V3,QT_Interval_ms,2198,2081,2144,2026,172,0.921747
10,f_1000000_s42,V6,T_Amp,2198,2164,2180,2056,142,0.935396
4,f_1000000_s42,V3,T_Amp,2198,2126,2163,2080,118,0.946315
7,f_1000000_s42,V6,Q_Amp,2198,2167,2180,2122,76,0.965423
1,f_1000000_s42,V3,Q_Amp,2198,2167,2189,2149,49,0.977707
0,f_1000000_s42,V3,P_Amp,2198,2178,2186,2154,44,0.979982
9,f_1000000_s42,V6,S_Amp,2198,2197,2198,2155,43,0.980437
3,f_1000000_s42,V3,S_Amp,2198,2182,2191,2173,25,0.988626
2,f_1000000_s42,V3,R_Amp,2198,2183,2191,2177,21,0.990446


,model_id,lead,clinical_feature,pairing_tolerance_ms,median_abs_pairing_error_ms,p95_abs_pairing_error_ms,n_unmatched_real_events,n_unmatched_recon_events,unmatched_event_fraction
10,f_1000000_s42,V6,T_Amp,100.0,6.0,64.0,6773,7893,0.307000
11,f_1000000_s42,V6,QT_Interval_ms,100.0,22.0,88.0,4789,4695,0.241446
7,f_1000000_s42,V6,Q_Amp,100.0,10.0,64.0,4264,3525,0.173842
5,f_1000000_s42,V3,QT_Interval_ms,100.0,18.0,84.0,2183,4428,0.156967
9,f_1000000_s42,V6,S_Amp,100.0,18.0,88.0,3381,3413,0.130034
1,f_1000000_s42,V3,Q_Amp,100.0,18.0,66.0,1555,3160,0.096384
4,f_1000000_s42,V3,T_Amp,100.0,6.0,34.0,1837,2592,0.093607
0,f_1000000_s42,V3,P_Amp,100.0,8.0,54.0,1989,2054,0.081652
6,f_1000000_s42,V6,P_Amp,100.0,16.0,66.0,1815,1970,0.074480
3,f_1000000_s42,V3,S_Amp,100.0,4.0,14.0,800,894,0.032691


In [20]:
#| label: temporal-morphology-per-record-ledger
#| tbl-cap: Per-record detector states and patient-clustered paired feature differences for the first accepted exact model. Intervals are descriptive percentile-bootstrap intervals.
if temporal_models.empty:
    display(pd.DataFrame({"status": ["No accepted per-record ledger"]}))
else:
    anchor_model_id = temporal_models.sort_values("model_id").iloc[0].model_id
    anchor_metadata = json.loads((temporal_root / f"{anchor_model_id}.json").read_text())
    anchor_per_record_path = temporal_root / anchor_metadata["per_record_parquet"]
    assert hashlib.sha256(anchor_per_record_path.read_bytes()).hexdigest() == (
        anchor_metadata["per_record_parquet_sha256"]
    )
    anchor_per_record = pd.read_parquet(anchor_per_record_path)
    assert len(anchor_per_record) == anchor_metadata["per_record_rows"] == 2198 * 2 * 6

    detector_strata = (
        anchor_per_record.groupby(
            ["lead", "clinical_feature", "detector_state"], as_index=False
        ).size()
    )
    detector_strata["record_fraction"] = detector_strata["size"] / 2198
    display(detector_strata.sort_values(
        ["lead", "clinical_feature", "size"], ascending=[True, True, False]
    ))

    ptb_patient_map = pd.read_csv(
        ROOT / "data/ptb_xl/ptbxl_database.csv", usecols=["ecg_id", "patient_id"]
    )
    paired_record = anchor_per_record.loc[
        anchor_per_record.n_finite_paired_events > 0
    ].copy()
    paired_record["ecg_id"] = paired_record.record_id.astype(int)
    paired_record = paired_record.merge(
        ptb_patient_map, on="ecg_id", how="left", validate="many_to_one"
    )
    assert paired_record.patient_id.notna().all()

    rng = np.random.default_rng(20260801)
    clustered_rows = []
    for (lead, feature), group in paired_record.groupby(["lead", "clinical_feature"]):
        patient_difference = group.groupby("patient_id").paired_mean_difference.mean()
        values = patient_difference.to_numpy(float)
        bootstrap = np.array([
            rng.choice(values, size=len(values), replace=True).mean()
            for _ in range(1000)
        ])
        clustered_rows.append({
            "model_id": anchor_model_id,
            "lead": lead,
            "clinical_feature": feature,
            "records_with_finite_pair": len(group),
            "patients_with_finite_pair": len(values),
            "equal_patient_mean_difference": values.mean(),
            "percentile_95_ci_low": np.percentile(bootstrap, 2.5),
            "percentile_95_ci_high": np.percentile(bootstrap, 97.5),
            "record_median_mae": group.paired_mae.median(),
            "record_p95_mae": group.paired_mae.quantile(0.95),
        })
    display(pd.DataFrame(clustered_rows))

,lead,clinical_feature,detector_state,size,record_fraction
0,V3,P_Amp,both_detected,2169,0.986806
2,V3,P_Amp,reconstruction_only,17,0.007734
3,V3,P_Amp,target_only,9,0.004095
1,V3,P_Amp,neither_detected,3,0.001365
4,V3,QT_Interval_ms,both_detected,2062,0.938126
6,V3,QT_Interval_ms,reconstruction_only,82,0.037307
5,V3,QT_Interval_ms,neither_detected,35,0.015924
7,V3,QT_Interval_ms,target_only,19,0.008644
8,V3,Q_Amp,both_detected,2159,0.982257
10,V3,Q_Amp,reconstruction_only,30,0.013649


,model_id,lead,clinical_feature,records_with_finite_pair,patients_with_finite_pair,equal_patient_mean_difference,percentile_95_ci_low,percentile_95_ci_high,record_median_mae,record_p95_mae
0,f_1000000_s42,V3,P_Amp,2154,1874,-0.116765,-0.119457,-0.114036,0.123050,0.261464
1,f_1000000_s42,V3,QT_Interval_ms,2026,1789,-12.634568,-14.490579,-10.660130,36.194444,113.000000
2,f_1000000_s42,V3,Q_Amp,2149,1865,-0.147352,-0.151881,-0.141839,0.161898,0.307866
3,f_1000000_s42,V3,R_Amp,2177,1888,-0.287234,-0.306183,-0.270133,0.262045,1.084522
4,f_1000000_s42,V3,S_Amp,2173,1886,-0.082102,-0.105695,-0.057213,0.371169,0.998482
5,f_1000000_s42,V3,T_Amp,2080,1831,-0.131216,-0.138585,-0.123886,0.133013,0.419968
6,f_1000000_s42,V6,P_Amp,2190,1901,-0.040031,-0.042421,-0.037963,0.066241,0.238697
7,f_1000000_s42,V6,QT_Interval_ms,2018,1777,-31.158919,-35.280660,-26.997911,67.732143,220.000000
8,f_1000000_s42,V6,Q_Amp,2122,1848,-0.032959,-0.036915,-0.028314,0.077631,0.256570
9,f_1000000_s42,V6,R_Amp,2195,1904,-0.258451,-0.277131,-0.241096,0.261301,1.066828


In [21]:
#| label: temporal-morphology-tolerance-sensitivity
#| fig-cap: Pairing-coverage sensitivity across five prespecified temporal windows for the first accepted exact model.
if temporal_models.empty:
    display(pd.DataFrame({"status": ["No accepted tolerance-sensitivity artifact"]}))
else:
    sensitivity_path = temporal_root / anchor_metadata["tolerance_sensitivity_csv"]
    assert hashlib.sha256(sensitivity_path.read_bytes()).hexdigest() == (
        anchor_metadata["tolerance_sensitivity_csv_sha256"]
    )
    tolerance_sensitivity = pd.read_csv(sensitivity_path)
    assert len(tolerance_sensitivity) == 2 * 6 * 5
    sensitivity_plot = px.line(
        tolerance_sensitivity,
        x="pairing_tolerance_ms",
        y="record_pair_coverage",
        color="clinical_feature",
        facet_col="lead",
        markers=True,
        hover_data=[
            "n_paired_events", "n_unmatched_real_events",
            "n_unmatched_recon_events", "variance_ratio",
        ],
        labels={
            "pairing_tolerance_ms": "Pairing tolerance (ms)",
            "record_pair_coverage": "Records with a matched event",
            "clinical_feature": "Feature",
        },
    )
    sensitivity_plot.update_yaxes(range=[0, 1])
    sensitivity_plot.update_layout(height=520)
    sensitivity_plot.show()

    endpoints = tolerance_sensitivity[
        tolerance_sensitivity.pairing_tolerance_ms.isin([25.0, 150.0])
    ].pivot(
        index=["lead", "clinical_feature"],
        columns="pairing_tolerance_ms",
        values=["record_pair_coverage", "variance_ratio", "n_paired_events"],
    )
    endpoints.columns = [f"{metric}_{int(tolerance)}ms" for metric, tolerance in endpoints.columns]
    endpoints = endpoints.reset_index()
    endpoints["coverage_delta_150_minus_25ms"] = (
        endpoints.record_pair_coverage_150ms - endpoints.record_pair_coverage_25ms
    )
    endpoints["variance_ratio_delta_150_minus_25ms"] = (
        endpoints.variance_ratio_150ms - endpoints.variance_ratio_25ms
    )
    endpoints["paired_event_delta_150_minus_25ms"] = (
        endpoints.n_paired_events_150ms - endpoints.n_paired_events_25ms
    )
    display(endpoints)

,lead,clinical_feature,record_pair_coverage_25ms,record_pair_coverage_150ms,variance_ratio_25ms,variance_ratio_150ms,n_paired_events_25ms,n_paired_events_150ms,coverage_delta_150_minus_25ms,variance_ratio_delta_150_minus_25ms,paired_event_delta_150_minus_25ms
0,V3,P_Amp,0.972702,0.982257,0.396361,0.372758,19461.0,23395.0,0.009554,-0.023603,3934.0
1,V3,QT_Interval_ms,0.866697,0.926752,1.025542,0.923160,10025.0,18618.0,0.060055,-0.102381,8593.0
2,V3,Q_Amp,0.936306,0.977707,0.403672,0.342238,13583.0,22529.0,0.041401,-0.061435,8946.0
3,V3,R_Amp,0.984531,0.990446,0.205778,0.214499,24857.0,25528.0,0.005914,0.008721,671.0
4,V3,S_Amp,0.979072,0.989081,0.154772,0.172240,24378.0,25198.0,0.010009,0.017467,820.0
5,V3,T_Amp,0.922657,0.949045,0.489344,0.495901,19834.0,21850.0,0.026388,0.006557,2016.0
6,V6,P_Amp,0.979982,0.998635,0.153826,0.096796,15996.0,24336.0,0.018653,-0.057029,8340.0
7,V6,QT_Interval_ms,0.794359,0.923112,1.749100,1.611685,7864.0,15838.0,0.128753,-0.137415,7974.0
8,V6,Q_Amp,0.890810,0.969063,0.113200,0.099291,12478.0,18913.0,0.078253,-0.013909,6435.0
9,V6,R_Amp,0.984531,0.998635,0.288752,0.306333,25106.0,26019.0,0.014104,0.017582,913.0


In [22]:
#| label: temporal-morphology-model-snapshot
#| tbl-cap: Descriptive model-level morphology snapshot for accepted artifacts. Masks differ in multiple loss factors, so this table is not a causal contrast.
if temporal_features.empty:
    display(pd.DataFrame({"status": ["No accepted model snapshots"]}))
else:
    model_snapshot = (
        temporal_features.groupby(["model_id", "model_mask", "mmd_kernel"], as_index=False)
        .agg(
            lead_feature_rows=("clinical_feature", "size"),
            minimum_pair_coverage=("record_pair_coverage", "min"),
            maximum_pair_coverage=("record_pair_coverage", "max"),
            median_variance_ratio=("variance_ratio", "median"),
            rows_below_unit_variance=("variance_ratio", lambda x: int((x < 1).sum())),
            maximum_variance_ratio=("variance_ratio", "max"),
            median_distribution_rbf_mmd2=("distribution_rbf_mmd2", "median"),
            maximum_distribution_rbf_mmd2=("distribution_rbf_mmd2", "max"),
        )
    )
    display(model_snapshot)

,model_id,model_mask,mmd_kernel,lead_feature_rows,minimum_pair_coverage,maximum_pair_coverage,median_variance_ratio,rows_below_unit_variance,maximum_variance_ratio,median_distribution_rbf_mmd2,maximum_distribution_rbf_mmd2
0,f_1000000_s42,1000000,0,12,0.918107,0.998635,0.259745,11,1.652626,0.104352,0.521378


In [23]:
#| label: temporal-morphology-kernel-2-vs-3
#| tbl-cap: Same-mask, same-seed descriptive contrast between MMD kernels 2 and 3. Aggregate detector-paired features are shown without inferential uncertainty.
matched_ids = {"f_1011012_s42", "f_1011013_s42"}
if matched_ids.issubset(set(temporal_features.model_id)):
    matched = temporal_features[temporal_features.model_id.isin(matched_ids)][[
        "model_id", "lead", "clinical_feature", "variance_ratio",
        "distribution_rbf_mmd2", "record_pair_coverage", "mean_recon",
    ]]
    kernel_2 = matched[matched.model_id.eq("f_1011012_s42")].drop(columns="model_id")
    kernel_3 = matched[matched.model_id.eq("f_1011013_s42")].drop(columns="model_id")
    kernel_contrast = kernel_2.merge(
        kernel_3,
        on=["lead", "clinical_feature"],
        suffixes=("_kernel2", "_kernel3"),
        validate="one_to_one",
    )
    kernel_contrast["variance_ratio_delta_kernel3_minus_kernel2"] = (
        kernel_contrast.variance_ratio_kernel3 - kernel_contrast.variance_ratio_kernel2
    )
    kernel_contrast["coverage_delta_kernel3_minus_kernel2"] = (
        kernel_contrast.record_pair_coverage_kernel3
        - kernel_contrast.record_pair_coverage_kernel2
    )
    kernel_contrast["distribution_mmd2_delta_kernel3_minus_kernel2"] = (
        kernel_contrast.distribution_rbf_mmd2_kernel3
        - kernel_contrast.distribution_rbf_mmd2_kernel2
    )
    display(kernel_contrast)
    display(pd.DataFrame([
        ("Lead-feature rows", len(kernel_contrast)),
        ("Rows with lower variance ratio under kernel 3", int((kernel_contrast.variance_ratio_delta_kernel3_minus_kernel2 < 0).sum())),
        ("Median variance-ratio delta, kernel 3 − kernel 2", kernel_contrast.variance_ratio_delta_kernel3_minus_kernel2.median()),
        ("Median distribution-MMD² delta, kernel 3 − kernel 2", kernel_contrast.distribution_mmd2_delta_kernel3_minus_kernel2.median()),
        ("Median pairing-coverage delta, kernel 3 − kernel 2", kernel_contrast.coverage_delta_kernel3_minus_kernel2.median()),
    ], columns=["Descriptive contrast", "Observed value"]))
else:
    display(pd.DataFrame({"status": ["Matched kernel-2/kernel-3 artifacts are not both accepted"]}))

,status
0,Matched kernel-2/kernel-3 artifacts are not bo...
